### Travel Planner

A travel planner agent workflow using MCP. It comprises of:
- A local Travel MCP server for traveler resources, itinerary planning, and simulated booking.
- External research MCP servers so the TravelResearcher can use Brave Search and browser-based page fetching.

In [ ]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from travel import reset_traveler
from travel_client import read_travel_itinerary, read_travel_preferences, read_travel_profile, list_travel_tools

load_dotenv(override=True)

Extra setup for my macbook

In [ ]:
import shutil

node_path = shutil.which("node") or "/usr/local/bin/node" 
node_dir = os.path.dirname(node_path)

# Update the PATH so npx can find 'node'
env = os.environ.copy()
env["PATH"] = f"{node_dir}:{env.get('PATH', '')}"
uv_command = "/opt/homebrew/bin/uv"
uvx_command = "/opt/homebrew/bin/uvx"
npx_command = shutil.which("npx") or "/usr/local/bin/npx"

mcp servers

In [ ]:
travel_mcp_server_params = [
    {"command": uv_command, "args": ["run", "travel_server.py"], "env": env}
]
travel_researcher_mcp_server_params = [
    {"command": uvx_command, "args": ["mcp-server-fetch"], "env": env},
    {"command": npx_command, "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": env},
]
travel_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=60) for params in travel_mcp_server_params]
travel_researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=60) for params in travel_researcher_mcp_server_params]


travel tools

In [ ]:
travel_tools = await list_travel_tools()
travel_tools


In [ ]:
traveler_name = "Yemi"
reset_traveler(traveler_name)

display(Markdown(await read_travel_profile(traveler_name)))
display(Markdown(await read_travel_preferences(traveler_name)))


travel researcher agent

In [ ]:
async def get_travel_researcher(mcp_servers) -> Agent:
    instructions = f"""You are a travel researcher. You can use Brave Search and browser-based page fetching to research the web.
Look for hotel, neighborhood, restaurant, and activity options that fit the traveler's destination, budget, interests, and pace.
Cross-check important details before recommending options, and keep your final research concise and practical.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    return Agent(
        name="TravelResearcher",
        instructions=instructions,
        model="gpt-4.1-mini",
        mcp_servers=mcp_servers,
    )


async def get_travel_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_travel_researcher(mcp_servers)
    return researcher.as_tool(
        tool_name="TravelResearcher",
        tool_description="Research live travel options on the web using Brave Search and browser-based page fetching.",
    )


In [ ]:
travel_profile = await read_travel_profile(traveler_name)
travel_preferences = await read_travel_preferences(traveler_name)

travel_instructions = f"""
You are a travel planner and booking assistant. The traveler you manage is {traveler_name}.
You have access to a TravelResearcher tool for web research and to local MCP tools that let you search hotels and activities,
save an itinerary, add itinerary items, and make bookings.
Your first goal is to create a trip that fits the traveler's preferences and budget.
Your second goal is to save the itinerary and complete the main bookings.
Traveler profile:
{travel_profile}
Traveler preferences:
{travel_preferences}
Do not ask for confirmation. Make sensible choices, stay within budget, and explain your decisions clearly.
"""

travel_prompt = """
Plan a 3-day Lagos birthday staycation for the traveler.
Use the TravelResearcher tool to browse the web and compare options before booking.
Then use the local travel MCP tools to save an itinerary, add itinerary items for each day, book one hotel, and book two activities.
Finish with a concise summary of the final plan, budget fit, and booked items.
"""


In [ ]:
for server in travel_mcp_servers + travel_researcher_mcp_servers:
    await server.connect()

travel_researcher_tool = await get_travel_researcher_tool(travel_researcher_mcp_servers)
travel_planner = Agent(
    name="TravelPlanner",
    instructions=travel_instructions,
    tools=[travel_researcher_tool],
    mcp_servers=travel_mcp_servers,
    model="gpt-4o-mini",
)

with trace("TravelPlanner"):
    travel_result = await Runner.run(travel_planner, travel_prompt, max_turns=30)

display(Markdown(travel_result.final_output))


In [ ]:
display(Markdown(await read_travel_itinerary(traveler_name)))
